# Configuring PyTorch for Native Windows GPUs

When transitioning from cloud-based environments (like Google Colab) to local development on a Windows machine, the biggest hurdle is getting the Python ecosystem to successfully communicate with the local NVIDIA GPU.

By default, standard Python package managers prioritize maximum compatibility over maximum performance, which often leads to "CUDA not available" errors. This notebook documents the exact architecture and commands required to build a GPU-accelerated local environment.

## 1. The Core Problem: The CPU Default

When starting a new project, developers typically run:
`pip install -r requirements.txt`

If that text file contains the package `torch`, pip will reach out to the default Python Package Index (PyPI).

**The Catch:** On Windows, the default `torch` package hosted on PyPI is the **CPU-only** version. It contains none of the NVIDIA drivers required for hardware acceleration. Even if you have a massive GPU and the latest NVIDIA drivers installed on your operating system, Python will be completely blind to it.

## 2. The Solution: Targeting the CUDA Index

To fix this, we have to bypass the default PyPI servers and download the binaries directly from PyTorch's custom hosting, specifically targeting our CUDA version (e.g., CUDA 12.1).

**Step 1: Purge the CPU Version**
First, we must completely remove the default packages to prevent conflicts.
Run this in the terminal (make sure the virtual environment for the project is activated):
```bash
pip uninstall torch torchvision torchaudio -y
```

** Step 2: Inject the CUDA Version
Next, we install the entire PyTorch ecosystem while pointing pip to the specific CUDA 12.1 index URL.
Run this in the terminal:
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

`(Note: This is a hefty download, usually around 2.5 GB, because it contains all the Nvidia drivers necessary to make the magic happen).`

In [1]:
# Checking if Pytorch has access to GPU:

import torch

# 1. Check if PyTorch can see the CUDA bridge
gpu_available = torch.cuda.is_available()
print(f"Is the GPU available? {gpu_available}")

# 2. Verify it is reading the correct hardware
if gpu_available:
    device_count = torch.cuda.device_count()
    device_name = torch.cuda.get_device_name(0)
    print(f"Total GPUs found: {device_count}")
    print(f"Primary GPU Name: {device_name}")
else:
    print("Warning: PyTorch is still defaulting to the CPU.")

Is the GPU available? True
Total GPUs found: 1
Primary GPU Name: NVIDIA GeForce RTX 4080 Laptop GPU
